## Trabalho Final: Construção de um Pipeline em PySpark

Este trabalho visa à reprodução de um pipeline de dados simplificado, utilizando como fonte uma base de dados proveniente do portal Dados Públicos (https://dados.gov.br/).

---

A base de dados designada para cada estudante foi previamente selecionada e está detalhada na seguinte [planilha](https://docs.google.com/spreadsheets/d/1mvl2edAtk--W4LuUeT8p-srYJTzdAww0/edit?usp=sharing&ouid=105817500216613802640&rtpof=true&sd=true). Nesta planilha, o estudante encontrará seu `Nome`, o `Nome da base` com a qual deverá trabalhar e o `link` para download e identificação do conjunto de dados.


## Orientações para o Desenvolvimento do Projeto

Desenvolva um script em Python, utilizando a biblioteca PySpark, que atenda aos seguintes requisitos:

1.  Realizar a leitura de um arquivo no formato CSV ou Parquet.
2.  Converter todos os valores vazios (empty values) para o tipo Nulo (NULL).
3.  Aplicar a tipagem de dados correta para *todas* as colunas do DataFrame.
4.  Definir e aplicar um valor *default* para *todas* as colunas do DataFrame, onde aplicável.
5.  Salvar o DataFrame resultante no formato Parquet, utilizando uma coluna específica para particionamento.
6.  O script desenvolvido deve ser executável por meio do comando `spark-submit`.

---

## Considerações Relevantes

1.  Priorize a aplicação dos conceitos e técnicas abordados em sala de aula.
2.  A avaliação do trabalho será fundamentada nos seis critérios estabelecidos na seção anterior, verificando a capacidade do código em implementar cada um dos tópicos.
3.  Caso opte por desenvolver o projeto em um notebook Python, é imprescindível a utilização do comando mágico `%%writefile` para a criação do arquivo `.py` final.

---

## Critérios de Avaliação e Políticas Acadêmicas

1.  Projetos que demonstrarem o uso de ferramentas de Inteligência Artificial para a geração de código terão sua nota zerada.
2.  Apresentação de trabalhos idênticos resultará na anulação da nota para todos os envolvidos.
3.  Trabalhos desenvolvidos em linguagens de programação diferentes de Python (e.g., Scala, R ou Java) não serão aceitos e terão sua nota zerada.


### Revisão

In [ ]:
!pip install pyspark duckdb

In [ ]:
from IPython.core.magic import (register_line_magic, register_cell_magic,register_line_cell_magic)

@register_line_cell_magic('sql')
def sparksql(line, cell=None):
    "Esse Magic funciona com  %sql e %%sql"
    try: spark
    except NameError: print('Spark instance is not defined')
    else:
        spark.conf.set('spark.sql.repl.eagerEval.enabled','true')
        spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 100)
        spark.conf.set('spark.sql.repl.eagerEval.truncate',-1)
        if cell is None:
            result = spark.sql(line)
            return result
        else:
            result = spark.sql(cell)
            return result

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf
import time

Config = SparkConf()
Config.set("spark.sql.repl.eagerEval.enabled", True)
Config.set("spark.sql.repl.eagerEval.maxNumRows", "20")
Config.set("spark.sql.repl.eagerEval.truncate", "-1")
Config.set("spark.driver.memory","10G")
Config.set("spark.memory.fraction", 0.9)
Config.set("spark.sql.adaptive.enabled", "true")
Config.set("spark.sql.adaptive.join.enabled", "true")
Config.set('spark.sql.legacy.allowNonEmptyLocationInCTAS','true')

# Config.set("spark.sql.shuffle.partitions", 100)
# Config.set("spark.default.parallelism", 200)

spark = SparkSession.builder.config(conf=Config).master("local[*]").appName("TrabalhoFinal").getOrCreate()

In [ ]:
df = spark.read.option("header", True).option("delimiter",";").csv("/content/D.SDA.PDA.005.CAT.202305.csv")
df.show(10)

+--------------------------+--------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+---------------------+--------------------+--------------------+---------------------------+--------------------+-------------+----------------+--------------------+--------------------+---------------+-----------------------+---------------+---------------+----------------+-------------------+
|Agente  Causador  Acidente|Data Acidente1|                 CBO|              CID-10|CNAE2.0 Empregador4| CNAE2.0 Empregador5|       Emitente CAT|Esp�cie do benef�cio|   Filia��o Segurado|Indica �bito Acidente|          Munic Empr|   Natureza da Les�o|Origem de Cadastramento CAT|Parte Corpo Atingida|         Sexo|Tipo do Acidente|UF  Munic.  Acidente|UF Munic. Empregador|Data Acidente18|Data Despacho Benef�cio|Data Acidente20|Data Nascimento|Data Emiss�o CAT|CNPJ/CEI Empregador|
+--------------------------+----

In [ ]:
df.printSchema()

root
 |-- Agente  Causador  Acidente: string (nullable = true)
 |-- Data Acidente1: string (nullable = true)
 |-- CBO: string (nullable = true)
 |-- CID-10: string (nullable = true)
 |-- CNAE2.0 Empregador4: string (nullable = true)
 |-- CNAE2.0 Empregador5: string (nullable = true)
 |-- Emitente CAT: string (nullable = true)
 |-- Esp�cie do benef�cio: string (nullable = true)
 |-- Filia��o Segurado: string (nullable = true)
 |-- Indica �bito Acidente: string (nullable = true)
 |-- Munic Empr: string (nullable = true)
 |-- Natureza da Les�o: string (nullable = true)
 |-- Origem de Cadastramento CAT: string (nullable = true)
 |-- Parte Corpo Atingida: string (nullable = true)
 |-- Sexo: string (nullable = true)
 |-- Tipo do Acidente: string (nullable = true)
 |-- UF  Munic.  Acidente: string (nullable = true)
 |-- UF Munic. Empregador: string (nullable = true)
 |-- Data Acidente18: string (nullable = true)
 |-- Data Despacho Benef�cio: string (nullable = true)
 |-- Data Acidente20: stri

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

schema = StructType([
    StructField("Agente  Causador  Acidente", StringType(), True),
    StructField("Data Acidente1", DateType(), True),
    StructField("CBO", StringType(), True),
    StructField("CID-10", StringType(), True),
    StructField("CNAE2.0 Empregador4", StringType(), True),
    StructField("CNAE2.0 Empregador5", StringType(), True),
    StructField("Emitente CAT", StringType(), True),
    StructField("Esp cie do benef cio", StringType(), True),
    StructField("Filia  o Segurado", StringType(), True),
    StructField("Indica  bito Acidente", StringType(), True),
    StructField("Munic Empr", StringType(), True),
    StructField("Natureza da Les o", StringType(), True),
    StructField("Origem de Cadastramento CAT", StringType(), True),
    StructField("Parte Corpo Atingida", StringType(), True),
    StructField("Sexo", StringType(), True),
    StructField("Tipo do Acidente", StringType(), True),
    StructField("UF  Munic.  Acidente", StringType(), True),
    StructField("UF Munic. Empregador", StringType(), True),
    StructField("Data Acidente18", DateType(), True),
    StructField("Data Despacho Benef cio", DateType(), True),
    StructField("Data Acidente20", DateType(), True),
    StructField("Data Nascimento", DateType(), True),
    StructField("Data Emiss o CAT", DateType(), True),
    StructField("CNPJ/CEI Empregador", StringType(), True)
])

In [ ]:
df = spark.read.option("header", True).option("delimiter", ";").schema(schema).csv("/content/D.SDA.PDA.005.CAT.202305.csv")
df.show(5)
df.printSchema()

+--------------------------+--------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+---------------------+--------------------+--------------------+---------------------------+--------------------+-------------+----------------+--------------------+--------------------+---------------+-----------------------+---------------+---------------+----------------+-------------------+
|Agente  Causador  Acidente|Data Acidente1|                 CBO|              CID-10|CNAE2.0 Empregador4| CNAE2.0 Empregador5|       Emitente CAT|Esp cie do benef cio|   Filia  o Segurado|Indica  bito Acidente|          Munic Empr|   Natureza da Les o|Origem de Cadastramento CAT|Parte Corpo Atingida|         Sexo|Tipo do Acidente|UF  Munic.  Acidente|UF Munic. Empregador|Data Acidente18|Data Despacho Benef cio|Data Acidente20|Data Nascimento|Data Emiss o CAT|CNPJ/CEI Empregador|
+--------------------------+----

In [ ]:
df = df.replace("", None)
df.show(50)

+--------------------------+--------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+---------------------+--------------------+--------------------+---------------------------+--------------------+-------------+----------------+--------------------+--------------------+---------------+-----------------------+---------------+---------------+----------------+-------------------+
|Agente  Causador  Acidente|Data Acidente1|                 CBO|              CID-10|CNAE2.0 Empregador4| CNAE2.0 Empregador5|       Emitente CAT|Esp cie do benef cio|   Filia  o Segurado|Indica  bito Acidente|          Munic Empr|   Natureza da Les o|Origem de Cadastramento CAT|Parte Corpo Atingida|         Sexo|Tipo do Acidente|UF  Munic.  Acidente|UF Munic. Empregador|Data Acidente18|Data Despacho Benef cio|Data Acidente20|Data Nascimento|Data Emiss o CAT|CNPJ/CEI Empregador|
+--------------------------+----

In [ ]:
import re

def limpar_nome(col):
    col = re.sub(r"[^\w\s]", "", col)
    col = col.strip().replace(" ", "_")
    col = re.sub(r"_+", "_", col)
    return col

for old in df.columns:
    df = df.withColumnRenamed(old, limpar_nome(old))

print(df.columns)

['Agente_Causador_Acidente', 'Data_Acidente1', 'CBO', 'CID10', 'CNAE20_Empregador4', 'CNAE20_Empregador5', 'Emitente_CAT', 'Esp_cie_do_benef_cio', 'Filia_o_Segurado', 'Indica_bito_Acidente', 'Municipio_Empregador', 'Natureza_da_Les_o', 'Origem_de_Cadastramento_CAT', 'Parte_Corpo_Atingida', 'Sexo', 'Tipo_do_Acidente', 'UF_Munic_Acidente', 'UF_Munic_Empregador', 'Data_Acidente18', 'Data_Despacho_Benef_cio', 'Data_Acidente20', 'Data_Nascimento', 'Data_Emiss_o_CAT', 'CNPJCEI_Empregador']


In [ ]:
df = df.fillna({
    "Sexo": "Não informado",
    "Municipio_Empregador": "Não informado",
    "UF_Munic_Acidente": "Não informado",
    "UF_Munic_Empregador": "Não informado"
})

df.show(10)

+------------------------+--------------+--------------------+--------------------+------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------+-------------+----------------+-----------------+-------------------+---------------+-----------------------+---------------+---------------+----------------+------------------+
|Agente_Causador_Acidente|Data_Acidente1|                 CBO|               CID10|CNAE20_Empregador4|  CNAE20_Empregador5|       Emitente_CAT|Esp_cie_do_benef_cio|    Filia_o_Segurado|Indica_bito_Acidente|Municipio_Empregador|   Natureza_da_Les_o|Origem_de_Cadastramento_CAT|Parte_Corpo_Atingida|         Sexo|Tipo_do_Acidente|UF_Munic_Acidente|UF_Munic_Empregador|Data_Acidente18|Data_Despacho_Benef_cio|Data_Acidente20|Data_Nascimento|Data_Emiss_o_CAT|CNPJCEI_Empregador|
+------------------------+--------------+---------

In [ ]:
from pyspark.sql.functions import col, sum

valores_nulos = df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])

valores_nulos.show()

+------------------------+--------------+---+-----+------------------+------------------+------------+--------------------+----------------+--------------------+--------------------+-----------------+---------------------------+--------------------+----+----------------+-----------------+-------------------+---------------+-----------------------+---------------+---------------+----------------+------------------+
|Agente_Causador_Acidente|Data_Acidente1|CBO|CID10|CNAE20_Empregador4|CNAE20_Empregador5|Emitente_CAT|Esp_cie_do_benef_cio|Filia_o_Segurado|Indica_bito_Acidente|Municipio_Empregador|Natureza_da_Les_o|Origem_de_Cadastramento_CAT|Parte_Corpo_Atingida|Sexo|Tipo_do_Acidente|UF_Munic_Acidente|UF_Munic_Empregador|Data_Acidente18|Data_Despacho_Benef_cio|Data_Acidente20|Data_Nascimento|Data_Emiss_o_CAT|CNPJCEI_Empregador|
+------------------------+--------------+---+-----+------------------+------------------+------------+--------------------+----------------+--------------------+---

In [ ]:
df = df.fillna({
    "Data_Acidente1": "1900-01-01",
    "Data_Acidente18": "1900-01-01",
    "Data_Acidente20": "1900-01-01",
    "Data_Despacho_Benef_cio": "1900-01-01",
    "Data_Nascimento": "1900-01-01",
    "Data_Emiss_o_CAT": "1900-01-01"
})

In [ ]:
df.write.mode("overwrite").partitionBy("UF_Munic_Acidente").parquet("/content/parquet_output")

In [ ]:
import os

base_path = "/content/parquet_output"

print(os.listdir(base_path)[:10])

['UF_Munic_Acidente=2023%2F05', 'UF_Munic_Acidente=Não informado', 'UF_Munic_Acidente=Maranh�o        ', '._SUCCESS.crc', 'UF_Munic_Acidente=Rio Grande Norte', 'UF_Munic_Acidente=0000%2F00', 'UF_Munic_Acidente=Empregador         ', 'UF_Munic_Acidente=Piau�           ', 'UF_Munic_Acidente=Alagoas         ', 'UF_Munic_Acidente=08%2F05%2F2023']


In [ ]:
%%writefile cat_pipeline.py
from pyspark.sql import SparkSession
import re

def limpar_nome(col):
    col = re.sub(r"[^\w\s]", "", col)
    col = col.strip().replace(" ", "_")
    col = re.sub(r"_+", "_", col)
    return col

def main():
    spark = SparkSession.builder.appName("CAT_Pipeline").getOrCreate()

    df = spark.read.option("header", True).option("delimiter", ";").csv("/content/D.SDA.PDA.005.CAT.202305.csv")

    df = df.replace("", None)

    for old in df.columns:
        df = df.withColumnRenamed(old, limpar_nome(old))

    df = df.fillna({
        "Data_Acidente1": "1900-01-01",
        "Data_Acidente18": "1900-01-01",
        "Data_Acidente20": "1900-01-01",
        "Data_Despacho_Benef_cio": "1900-01-01",
        "Data_Nascimento": "1900-01-01",
        "Data_Emiss_o_CAT": "1900-01-01"
    })


    df.write.mode("overwrite").partitionBy("UF_Munic_Acidente").parquet("/content/parquet_output")

    spark.stop()

if __name__ == "__main__":
    main()

Writing cat_pipeline.py
